<style>
  .nvidia-banner {background: linear-gradient(100deg,#0b0b0b,#292929); color:white;
                  border-left:10px solid #76b900; padding:18px 22px; margin:8px 0 18px;}
  .nvidia-banner h1 {margin:0 0 6px; font-size:30px;}
  .task {border-left:6px solid #76b900; background:#f5f8f1; padding:12px 16px; margin:12px 0;}
  .checkpoint {border:1px solid #b8b8b8; border-radius:6px; padding:10px 14px; background:#fafafa;}
  .warning {border-left:6px solid #f2a900; background:#fff8e6; padding:12px 16px;}
  code {font-size: 0.92em;}
</style>

<div class="nvidia-banner">
  <h1>Module 1 — Direct nvMolKit: Map ReFRAME Chemical Space</h1>
  <div>ACS Fall 2026 · Hands-on GPU cheminformatics · 45–60 minutes</div>
</div>

## Goal

You will write and modify a small nvMolKit workflow that turns a ReFRAME compound library into fingerprints, similarity neighborhoods, and chemical clusters. Most tasks require copying an existing line and changing a function parameter.

By the end, you can explain what fingerprint radius, bit length, and clustering cutoff change—and what the resulting clusters do **not** prove.


## Setup

A workshop image should already contain nvMolKit, RDKit, PyTorch, pandas, and matplotlib. nvMolKit 0.5 requires a compatible NVIDIA GPU and CUDA-enabled PyTorch; the project README currently recommends conda-forge and an explicitly compatible CUDA version, for example:

```bash
conda install -c conda-forge nvmolkit pytorch-gpu cuda-version=12.6 pandas matplotlib jupyter
```

Run the next cell. In the workshop environment, confirm that it prints `NVMOLKIT_READY == True` in the following status cell. A CPU fallback lets instructors inspect the notebook elsewhere, but it is not an nvMolKit performance demonstration.


This notebook imports shared ReFRAME loading and descriptor helpers from `workshop_common.py`; keep that file and `data/reframe_teaching_snapshot.csv` beside the workshop notebooks.

In [ ]:
# Prepare methods for describing, comparing, and grouping molecular structures.
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from rdkit import DataStructs, RDLogger, rdBase
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from workshop_common import (
    add_descriptors,
    load_reframe,
    require_bounded_condensed_distances,
    require_memory_within_limit,
    square_matrix_bytes,
)

MODULE1_STARTED = time.perf_counter()
RDLogger.DisableLog("rdApp.error")
SEED = 2026
np.random.seed(SEED)

# Use nvMolKit for batched chemical comparisons when a compatible GPU is available.
NVMOLKIT_READY = False
NVMOLKIT_IMPORT_ERROR = None
try:
    import torch
    import nvmolkit
    from nvmolkit.clustering import fused_butina
    from nvmolkit.fingerprints import MorganFingerprintGenerator
    from nvmolkit.similarity import crossTanimotoSimilarity

    NVMOLKIT_READY = bool(torch.cuda.is_available())
except Exception as exc:
    NVMOLKIT_IMPORT_ERROR = repr(exc)

print(f"RDKit {rdBase.rdkitVersion}")
if NVMOLKIT_READY:
    ACTIVE_BACKEND = "nvMolKit GPU"
    print(
        f"nvMolKit {nvmolkit.__version__} | CUDA devices: {torch.cuda.device_count()}"
    )
else:
    ACTIVE_BACKEND = "RDKit CPU fallback (not GPU evidence)"
    print(
        "CPU teaching fallback active: nvMolKit GPU calls will be shown but evaluated with RDKit."
    )
    print("Reason:", NVMOLKIT_IMPORT_ERROR or "torch.cuda.is_available() is False")

In [ ]:
def fingerprint_tensor(result):
    """Return the CUDA torch tensor wrapped by an nvMolKit fingerprint result."""
    return result if isinstance(result, torch.Tensor) else result.torch()


def gpu_to_numpy(result):
    """Synchronize an nvMolKit result or CUDA tensor and return a host array."""
    if isinstance(result, torch.Tensor):
        return result.detach().cpu().numpy()
    return result.numpy()


# Morgan fingerprints summarize the local atomic environments in each molecule.
def make_fingerprints(molecules, radius=2, fp_bits=1024):
    """Return nvMolKit CUDA fingerprints, or RDKit fingerprints in fallback mode."""
    if NVMOLKIT_READY:
        generator = MorganFingerprintGenerator(radius=radius, fpSize=fp_bits)
        return fingerprint_tensor(
            generator.GetFingerprints(list(molecules), num_threads=0)
        )
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_bits)
    return generator.GetFingerprints(list(molecules), numThreads=0)


# Tanimoto similarity measures how many fingerprint features two molecules share.
def tanimoto_matrix(first, second=None):
    """Return a host similarity matrix from the active backend."""
    if NVMOLKIT_READY:
        return gpu_to_numpy(crossTanimotoSimilarity(first, second))
    second = first if second is None else second
    return np.asarray(
        [DataStructs.BulkTanimotoSimilarity(query, second) for query in first],
        dtype=float,
    )


def time_call(operation, synchronize=None):
    """Measure one operation, synchronizing asynchronous CUDA work when requested."""
    if synchronize:
        synchronize()
    started = time.perf_counter()
    result = operation()
    if synchronize:
        synchronize()
    return result, time.perf_counter() - started


print("Shared ReFRAME helpers and Module 1 runtime helpers are ready.")

### What the notebook computes

- **Morgan fingerprints** encode circular atom environments as a fixed-length bit vector. Radius changes how far each local environment extends; fingerprint length changes the collision budget.
- **Tanimoto similarity** compares two binary fingerprints. It is a structural-neighborhood measure, not a measurement of target binding or biological activity.
- **Butina clustering** groups molecules using a distance cutoff. Here, `distance = 1 - Tanimoto similarity`, so a cutoff of `0.55` corresponds to a similarity threshold of `0.45`.

The notebook uses nvMolKit when a compatible NVIDIA GPU is available. Its CPU fallback exists only so the teaching narrative and checks remain inspectable on a non-GPU laptop; the workshop's accelerated exercises should report `NVMOLKIT_READY == True`.


## Step 1 — Load and quality-check ReFRAME

The default lesson uses the bundled 96-compound teaching snapshot, with no network access. It keeps classroom timing deterministic and includes three useful anchor compounds.

<div class="task"><b>Your edit:</b> Keep the default snapshot for the reference run. The separate advanced cell later in this notebook makes a live 10,000-row run explicit and shows its memory estimate first.</div>


In [ ]:
# Build a reproducible ReFRAME sample containing the named reference compounds.
DATA_SOURCE = "snapshot"
SAMPLE_SIZE = 96
ANCHOR_TERMS = ("imatinib", "linezolid", "ritonavir")

reframe = load_reframe(
    sample_size=SAMPLE_SIZE, anchor_terms=ANCHOR_TERMS, source=DATA_SOURCE
)
print("Data source:", reframe.attrs["source"])
print(f"Valid unique structures: {len(reframe):,}")
print("Invalid SMILES removed:", reframe.attrs["invalid_count"])
display(reframe[["name", "canonical_ikey", "status", "source"]].head(8))

### Check the library's physicochemical range

RDKit supplies lightweight descriptors; nvMolKit accelerates the batched operations that follow. Descriptor plots help us notice when a selection method accidentally narrows the chemical-property range.


In [ ]:
# Describe the library's range of molecular size, lipophilicity, and polarity.
characterized = add_descriptors(reframe)
summary = (
    characterized[["MolWt", "cLogP", "TPSA", "HBD", "HBA", "RotB"]].describe().round(2)
)
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for axis, column, color in zip(
    axes, ["MolWt", "cLogP", "TPSA"], ["#76b900", "#333333", "#5b8c00"]
):
    axis.hist(characterized[column], bins=30, color=color, alpha=0.85)
    axis.set(title=column, xlabel=column, ylabel="Compounds")
fig.suptitle("ReFRAME teaching sample: descriptor distributions", y=1.04)
plt.tight_layout()
plt.show()

## Step 2 — Generate Morgan fingerprints and compare runtime

nvMolKit mirrors the RDKit concept but processes a **list of molecules as a batch**. The returned packed fingerprints remain GPU-resident until you request host data.

The next cell computes the same Morgan fingerprints with both libraries. The timed region excludes generator construction. For nvMolKit, it includes CPU preprocessing and GPU execution, with a warm-up and `torch.cuda.synchronize()` around the measurement so asynchronous CUDA work is actually included. It does not include copying the packed fingerprints back to host memory.

<div class="task"><b>Your edits:</b>
<ol>
<li>Run with radius 2, then change only <code>FP_RADIUS</code> to 3. Compare both runtime and neighbor rankings.</li>
<li>Compare <code>FP_BITS = 1024</code> with <code>2048</code>.</li>
<li>The full Step 1 sample is the fingerprint batch. Change <code>SAMPLE_SIZE</code>, confirm the printed <code>FINGERPRINT_BATCH_SIZE</code>, and ask: at what batch size does nvMolKit's throughput advantage become clear on your GPU?</li>
</ol>
</div>

Treat these as classroom measurements, not publication-quality benchmarks: GPU model, software versions, contention, and repeated-run variability all matter.


In [ ]:
# Radius sets the chemical neighborhood captured around each atom.
FP_RADIUS = 2  # Exercise: change to 3
FP_BITS = 1024  # Exercise: compare with 2048
molecules = characterized["_mol"].tolist()
FINGERPRINT_BATCH_SIZE = len(molecules)
print(
    f"Fingerprint batch size: {FINGERPRINT_BATCH_SIZE:,} molecules (the full Step 1 sample)"
)

# Create the same molecular representation with RDKit and nvMolKit for comparison.
rdkit_fp_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS, fpSize=FP_BITS
)
rdkit_fingerprints, rdkit_fp_seconds = time_call(
    lambda: rdkit_fp_generator.GetFingerprints(molecules, numThreads=0)
)

runtime_rows = [
    {
        "backend": "RDKit CPU",
        "seconds": rdkit_fp_seconds,
        "molecules_per_second": len(molecules) / rdkit_fp_seconds,
    }
]

if NVMOLKIT_READY:
    fp_generator = MorganFingerprintGenerator(radius=FP_RADIUS, fpSize=FP_BITS)
    warmup_result = fp_generator.GetFingerprints(
        molecules[: min(32, len(molecules))], num_threads=0
    )
    _ = gpu_to_numpy(fingerprint_tensor(warmup_result))

    fingerprint_result, nvmolkit_fp_seconds = time_call(
        lambda: fp_generator.GetFingerprints(molecules, num_threads=0),
        synchronize=torch.cuda.synchronize,
    )
    fingerprints = fingerprint_tensor(fingerprint_result)
    packed_shape = tuple(fingerprints.shape)
    runtime_rows.append(
        {
            "backend": "nvMolKit GPU",
            "seconds": nvmolkit_fp_seconds,
            "molecules_per_second": len(molecules) / nvmolkit_fp_seconds,
        }
    )
else:
    fp_generator = rdkit_fp_generator
    fingerprints = rdkit_fingerprints
    packed_shape = (len(fingerprints), FP_BITS)

print("Fingerprint representation shape used downstream:", packed_shape)

fingerprint_runtime = pd.DataFrame(runtime_rows)
display(fingerprint_runtime.round({"seconds": 4, "molecules_per_second": 1}))
if NVMOLKIT_READY:
    print(
        f"Observed fingerprint speed ratio (RDKit / nvMolKit): {rdkit_fp_seconds / nvmolkit_fp_seconds:.2f}×"
    )
else:
    print("nvMolKit timing requires the workshop's compatible NVIDIA GPU environment.")

## Step 3 — Search structural neighborhoods and compare runtime

We will compare three anchor drugs against the whole sample in one cross-similarity calculation. The top hit is normally the anchor itself; we exclude it to expose its neighborhood.

The timed region starts from already-computed fingerprints and measures the same `3 × N` Tanimoto workload. The nvMolKit measurement uses a warm-up and CUDA synchronization; transfer of the completed similarity matrix to host memory is excluded from both timed regions.

<div class="task"><b>Your edits:</b>
<ol>
<li>Record RDKit and nvMolKit comparisons per second at the current sample size.</li>
<li>Change <code>SAMPLE_SIZE</code> in Step 1 and rerun through this cell. How does the speed ratio change?</li>
<li>After changing radius from 2 to 3, compare both runtime and top-five membership.</li>
</ol>
</div>


In [ ]:
# Locate the three reference compounds used to explore structural neighborhoods.
anchor_rows = []
for term in ANCHOR_TERMS:
    match = characterized[
        characterized["name"].str.contains(term, case=False, regex=False)
    ]
    if not match.empty:
        anchor_rows.append(match.iloc[0])
anchors = pd.DataFrame(anchor_rows).reset_index(drop=True)
anchor_molecules = anchors["_mol"].tolist()
comparison_count = len(anchor_molecules) * len(molecules)

rdkit_anchor_fps = rdkit_fp_generator.GetFingerprints(anchor_molecules, numThreads=0)
rdkit_similarity_rows, rdkit_similarity_seconds = time_call(
    lambda: [
        DataStructs.BulkTanimotoSimilarity(query, rdkit_fingerprints)
        for query in rdkit_anchor_fps
    ]
)
rdkit_similarity = np.asarray(rdkit_similarity_rows, dtype=float)

similarity_runtime_rows = [
    {
        "backend": "RDKit CPU",
        "comparisons": comparison_count,
        "seconds": rdkit_similarity_seconds,
        "comparisons_per_second": comparison_count / rdkit_similarity_seconds,
    }
]

if NVMOLKIT_READY:
    anchor_fps = fingerprint_tensor(
        fp_generator.GetFingerprints(anchor_molecules, num_threads=0)
    )
    warmup_library_fps = fingerprint_tensor(
        fp_generator.GetFingerprints(
            molecules[: min(32, len(molecules))], num_threads=0
        )
    )
    warmup_anchor_fps = fingerprint_tensor(
        fp_generator.GetFingerprints(anchor_molecules[:1], num_threads=0)
    )
    _ = gpu_to_numpy(crossTanimotoSimilarity(warmup_anchor_fps, warmup_library_fps))

    similarity_result, nvmolkit_similarity_seconds = time_call(
        lambda: crossTanimotoSimilarity(anchor_fps, fingerprints),
        synchronize=torch.cuda.synchronize,
    )
    similarity = gpu_to_numpy(similarity_result)
    similarity_runtime_rows.append(
        {
            "backend": "nvMolKit GPU",
            "comparisons": comparison_count,
            "seconds": nvmolkit_similarity_seconds,
            "comparisons_per_second": comparison_count / nvmolkit_similarity_seconds,
        }
    )
else:
    anchor_fps = rdkit_anchor_fps
    similarity = rdkit_similarity

# Rank ReFRAME compounds by structural similarity to each reference compound.
neighbor_rows = []
for query_index, anchor in anchors.iterrows():
    ranking = np.argsort(-similarity[query_index])
    ranking = [
        idx
        for idx in ranking
        if characterized.iloc[idx]["canonical_ikey"] != anchor["canonical_ikey"]
    ]
    for rank, library_index in enumerate(ranking[:5], start=1):
        neighbor_rows.append(
            {
                "query": anchor["name"].split(";")[0],
                "rank": rank,
                "neighbor": characterized.iloc[library_index]["name"],
                "tanimoto": float(similarity[query_index, library_index]),
                "profile": characterized.iloc[library_index]["reframedb_url"],
            }
        )

# Show the chemical neighbors before reporting the runtime comparison.
neighbors = pd.DataFrame(neighbor_rows)
display(neighbors.round({"tanimoto": 3}))

similarity_runtime = pd.DataFrame(similarity_runtime_rows)
display(similarity_runtime.round({"seconds": 6, "comparisons_per_second": 0}))
if NVMOLKIT_READY:
    print(
        f"Observed similarity speed ratio (RDKit / nvMolKit): {rdkit_similarity_seconds / nvmolkit_similarity_seconds:.2f}×"
    )
else:
    print(
        "nvMolKit similarity timing requires the workshop's compatible NVIDIA GPU environment."
    )

<div class="checkpoint">
<b>Interpretation checkpoint</b><br>
1. Which backend completed more comparisons per second, and how did sample size affect the ratio?<br>
2. Which anchor has the tightest local neighborhood?<br>
3. After changing radius from 2 to 3, which top-five members moved?<br>
4. Why would “similar structure” still be insufficient evidence for a repurposing claim?
</div>


## Step 4 — Cluster chemical space and compare runtime

`fused_butina` computes fingerprint similarities as clustering needs them, avoiding an `N × N` matrix. Its cutoff is a **distance** cutoff. The conventional RDKit workflow below first constructs a condensed Tanimoto-distance vector and then calls Butina clustering.

This is an end-to-end **fingerprints-to-cluster-labels** comparison. It intentionally compares the practical workflows rather than pretending the underlying implementations perform identical work. Fingerprint generation is excluded; distance/similarity evaluation and clustering are included.

<div class="task"><b>Your edits:</b>
<ol>
<li>Compare cutoffs 0.45, 0.55, and 0.65. A larger distance cutoff permits less-similar molecules to share a cluster.</li>
<li>For each cutoff, record runtime, number of clusters, singleton count, and largest-cluster size.</li>
<li>Change <code>SAMPLE_SIZE</code> and observe how the two workflows scale.</li>
</ol>
</div>


In [ ]:
# Butina clustering groups compounds with similar fingerprint patterns.
CLUSTER_DISTANCE_CUTOFF = 0.55  # Try 0.45 and 0.65


def rdkit_butina_from_fingerprints(rdkit_fps, distance_cutoff):
    """Run the conventional RDKit distance-vector and Butina workflow."""
    n_items = len(rdkit_fps)
    if n_items == 1:
        return np.array([0]), np.array([0])
    require_bounded_condensed_distances(n_items)
    distances = np.empty(n_items * (n_items - 1) // 2, dtype=np.float64)
    offset = 0
    for row in range(1, n_items):
        row_distances = DataStructs.BulkTanimotoSimilarity(
            rdkit_fps[row], rdkit_fps[:row], returnDistance=True
        )
        distances[offset : offset + row] = row_distances
        offset += row
    clusters = Butina.ClusterData(
        distances, n_items, distance_cutoff, isDistData=True, reordering=True
    )
    labels = np.full(n_items, -1, dtype=int)
    centroids = []
    for cluster_id, members in enumerate(clusters):
        labels[list(members)] = cluster_id
        centroids.append(members[0])
    return labels, np.asarray(centroids, dtype=int)


def nvmolkit_butina_from_fingerprints(fingerprint_matrix, distance_cutoff):
    """Convert nvMolKit 0.5.0 fused clusters to labels and centroid indices."""
    clusters, _, centroids = fused_butina(
        fingerprint_matrix, cutoff=distance_cutoff, return_centroids=True
    )
    labels = np.full(len(fingerprint_matrix), -1, dtype=int)
    for cluster_id, members in enumerate(clusters):
        labels[list(members)] = cluster_id
    centroids = np.asarray(centroids, dtype=int)
    assert (labels >= 0).all(), "Every molecule must receive a cluster label"
    assert len(centroids) == len(clusters), "Centroid and cluster counts must match"
    return labels, centroids


# Apply the same clustering idea with RDKit and nvMolKit.
rdkit_cluster_result, rdkit_clustering_seconds = time_call(
    lambda: rdkit_butina_from_fingerprints(rdkit_fingerprints, CLUSTER_DISTANCE_CUTOFF)
)
rdkit_cluster_ids, rdkit_centroid_indices = rdkit_cluster_result
clustering_runtime_rows = [
    {
        "backend": "RDKit CPU",
        "seconds": rdkit_clustering_seconds,
        "compounds_per_second": len(molecules) / rdkit_clustering_seconds,
        "clusters": len(np.unique(rdkit_cluster_ids)),
    }
]

if NVMOLKIT_READY:
    warmup_cluster_fps = fingerprint_tensor(
        fp_generator.GetFingerprints(
            molecules[: min(64, len(molecules))], num_threads=0
        )
    )
    _ = nvmolkit_butina_from_fingerprints(warmup_cluster_fps, CLUSTER_DISTANCE_CUTOFF)

    nvmolkit_cluster_result, nvmolkit_clustering_seconds = time_call(
        lambda: nvmolkit_butina_from_fingerprints(
            fingerprints, CLUSTER_DISTANCE_CUTOFF
        ),
        synchronize=torch.cuda.synchronize,
    )
    cluster_ids, centroid_indices = nvmolkit_cluster_result
    representative_kind = "nvMolKit fused Butina centroid"
    clustering_runtime_rows.append(
        {
            "backend": "nvMolKit GPU",
            "seconds": nvmolkit_clustering_seconds,
            "compounds_per_second": len(molecules) / nvmolkit_clustering_seconds,
            "clusters": len(np.unique(cluster_ids)),
        }
    )
else:
    cluster_ids = rdkit_cluster_ids
    centroid_indices = rdkit_centroid_indices
    representative_kind = "RDKit Butina centroid"

# Use one centroid as a representative of each structural family.
characterized["cluster_id"] = cluster_ids
cluster_sizes = characterized.groupby("cluster_id").size().sort_values(ascending=False)
print(f"Clusters used downstream: {len(cluster_sizes):,}")
print(f"Singleton clusters: {(cluster_sizes == 1).sum():,}")
print(f"Largest cluster: {cluster_sizes.iloc[0]:,} compounds")
print(f"Representative definition: {representative_kind}")

representatives = characterized.iloc[centroid_indices][
    ["name", "canonical_ikey", "MolWt", "cLogP", "TPSA", "cluster_id"]
].copy()
representatives["cluster_size"] = representatives["cluster_id"].map(cluster_sizes)
display(representatives.sort_values("cluster_size", ascending=False).head(12).round(2))

cluster_sizes.head(30).plot.bar(figsize=(11, 3), color="#76b900")
plt.title(f"Largest clusters (distance cutoff = {CLUSTER_DISTANCE_CUTOFF})")
plt.xlabel("Cluster ID")
plt.ylabel("Compounds")
plt.tight_layout()
plt.show()

# Report performance only after inspecting the resulting chemical families.
clustering_runtime = pd.DataFrame(clustering_runtime_rows)
display(clustering_runtime.round({"seconds": 4, "compounds_per_second": 1}))
if NVMOLKIT_READY:
    print(
        f"Observed clustering speed ratio (RDKit / nvMolKit): {rdkit_clustering_seconds / nvmolkit_clustering_seconds:.2f}×"
    )
else:
    print(
        "nvMolKit clustering timing requires the workshop's compatible NVIDIA GPU environment."
    )

## Advanced large run — explicit opt-in

The reference lesson above never uses the network and keeps RDKit CPU clustering at 96 rows. Set `ADVANCED_LARGE_RUN = True` only when you intentionally want the live ReFRAME export, have a compatible NVIDIA GPU, and have reviewed the printed 10,000-row float32 square-matrix estimate. The explicit 512 MiB guard applies to this GPU path; it is not a bound for RDKit CPU Butina.


In [ ]:
ADVANCED_LARGE_RUN = False
ADVANCED_SAMPLE_SIZE = 10_000
advanced_square_bytes = square_matrix_bytes(ADVANCED_SAMPLE_SIZE)

if ADVANCED_LARGE_RUN:
    print(f"10k float32 square matrix estimate: {advanced_square_bytes:,} bytes")
    require_memory_within_limit(advanced_square_bytes, limit_mib=512)
    if not NVMOLKIT_READY:
        raise RuntimeError(
            "Advanced live 10k run requires a compatible NVIDIA GPU and nvMolKit."
        )
    advanced_reframe = load_reframe(
        sample_size=ADVANCED_SAMPLE_SIZE,
        anchor_terms=ANCHOR_TERMS,
        source="live",
    )
    print("Advanced data source:", advanced_reframe.attrs["source"])
    print(f"Advanced rows: {len(advanced_reframe):,}")
    advanced_molecules = advanced_reframe["_mol"].tolist()
    advanced_generator = MorganFingerprintGenerator(radius=FP_RADIUS, fpSize=FP_BITS)
    advanced_fingerprint_result = advanced_generator.GetFingerprints(
        advanced_molecules, num_threads=0
    )
    advanced_fingerprints = fingerprint_tensor(advanced_fingerprint_result)
    advanced_cluster_result, advanced_seconds = time_call(
        lambda: nvmolkit_butina_from_fingerprints(
            advanced_fingerprints, CLUSTER_DISTANCE_CUTOFF
        ),
        synchronize=torch.cuda.synchronize,
    )
    advanced_cluster_ids, advanced_centroid_indices = advanced_cluster_result
    print(
        f"Advanced nvMolKit GPU clustering seconds={advanced_seconds:.3f}; "
        f"clusters={len(np.unique(advanced_cluster_ids)):,}; "
        f"centroids={len(advanced_centroid_indices):,}"
    )
else:
    print("Advanced live 10k run disabled; reference snapshot result is unchanged.")

## Checks and takeaways

Run the checks before moving to Module 2. If one fails, first confirm that all earlier cells were run in order.


In [ ]:
# Confirm that every molecule has a fingerprint, comparison, and cluster assignment.
assert len(characterized) > 0
fingerprint_count = int(fingerprints.shape[0]) if NVMOLKIT_READY else len(fingerprints)
assert fingerprint_count == len(characterized)
assert similarity.shape == (len(anchors), len(characterized))
assert len(cluster_ids) == len(characterized)
assert set(cluster_ids) == set(range(len(set(cluster_ids))))
assert neighbors["tanimoto"].between(0, 1).all()
assert fingerprint_runtime["seconds"].gt(0).all()
assert similarity_runtime["seconds"].gt(0).all()
assert clustering_runtime["seconds"].gt(0).all()
print(
    "✓ Data, results, cluster labels, and all three runtime tables are internally consistent."
)

MODULE1_REPORT = {
    "source": reframe.attrs["source"],
    "rows": int(len(characterized)),
    "backend": ACTIVE_BACKEND,
    "fingerprint_bits": int(FP_BITS),
    "elapsed_seconds": round(time.perf_counter() - MODULE1_STARTED, 6),
}
print("MODULE1_REPORT_JSON=" + json.dumps(MODULE1_REPORT, sort_keys=True))

### Takeaways

- Batch size is part of the method: the GPU is most useful when many molecules undergo the same operation.
- Fingerprint radius and clustering cutoff are scientific choices, not cosmetic settings.
- A diverse structural library is a starting point for screening design—not a prediction of activity.

**Save one observation:** Which parameter changed your scientific conclusion the most, and why?


## Sources and scientific boundary

- [nvMolKit repository](https://github.com/NVIDIA-BioNeMo/nvMolKit)
- [nvMolKit documentation](https://nvidia-bionemo.github.io/nvMolKit/)
- [Installed workshop nvMolKit skill](../skills/nvmolkit/SKILL.md) — authoritative for this environment
- [reframeDb](https://reframedb.org/) and its public `reframe_smiles_list.csv` export

The ReFRAME export is used for teaching and should be handled under the site's current terms. Refresh it before delivery and do not treat availability status as evidence of clinical suitability. Fingerprints, descriptors, clusters, and sampled force-field geometries do **not** establish binding, activity, ADMET, efficacy, safety, synthesizability, or experimental structure.


In [ ]:
# Repeat the neighbor search to test whether conclusions depend on fingerprint radius.
def _answer_key_neighbors(radius, top_k=5):
    """Return exact top-k non-self neighbors for the current sample and radius."""
    library_fps = make_fingerprints(molecules, radius=radius, fp_bits=FP_BITS)
    query_fps = make_fingerprints(
        anchors["_mol"].tolist(), radius=radius, fp_bits=FP_BITS
    )
    scores = tanimoto_matrix(query_fps, library_fps)
    rows = []
    for query_position, (_, anchor) in enumerate(anchors.iterrows()):
        order = np.argsort(-scores[query_position], kind="stable")
        order = [
            int(index)
            for index in order
            if characterized.iloc[index]["canonical_ikey"] != anchor["canonical_ikey"]
        ][:top_k]
        for rank, library_index in enumerate(order, start=1):
            candidate = characterized.iloc[library_index]
            rows.append(
                {
                    "radius": radius,
                    "query": anchor["name"].split(";")[0],
                    "rank": rank,
                    "neighbor": candidate["name"],
                    "neighbor_ikey": candidate["canonical_ikey"],
                    "tanimoto": float(scores[query_position, library_index]),
                }
            )
    return pd.DataFrame(rows)


answer_radius_2 = _answer_key_neighbors(radius=2)
answer_radius_3 = _answer_key_neighbors(radius=3)

tightness = (
    answer_radius_2.groupby("query")["tanimoto"]
    .agg(mean_top5="mean", weakest_top5="min", strongest_top5="max")
    .sort_values("mean_top5", ascending=False)
)
print("Tightest radius-2 neighborhood = highest mean top-five non-self similarity")
display(tightness.round(3))

# Compare neighbor membership rather than treating either radius as universally correct.
movement_rows = []
for query in answer_radius_2["query"].unique():
    radius_2_rows = answer_radius_2.query("query == @query")
    radius_3_rows = answer_radius_3.query("query == @query")
    radius_2 = dict(zip(radius_2_rows["neighbor_ikey"], radius_2_rows["neighbor"]))
    radius_3 = dict(zip(radius_3_rows["neighbor_ikey"], radius_3_rows["neighbor"]))
    movement_rows.append(
        {
            "query": query,
            "shared": len(set(radius_2) & set(radius_3)),
            "left_top5_at_radius_3": "; ".join(
                radius_2[key] for key in radius_2.keys() - radius_3.keys()
            )
            or "none",
            "entered_top5_at_radius_3": "; ".join(
                radius_3[key] for key in radius_3.keys() - radius_2.keys()
            )
            or "none",
        }
    )
neighbor_movement_answer = pd.DataFrame(movement_rows)
print("Exact top-five membership changes for this sample")
display(neighbor_movement_answer)

print("Measured similarity runtime for this environment")
display(similarity_runtime.round({"seconds": 6, "comparisons_per_second": 0}))

## Answer key — interpretation checkpoint

<details>
<summary><b>Reveal instructor key</b></summary>

1. **Runtime:** Use the Step 3 runtime table, which isolates the similarity calculation. On a compatible GPU and a sufficiently large batch, nvMolKit should usually complete more comparisons per second; its advantage should grow as the batch better amortizes launch and preprocessing overhead. RDKit may win for a small batch or a cold/contended GPU. Full credit requires reporting the observed numbers and recognizing that hardware and sample size control the answer.

2. **Tightest neighborhood:** The correct anchor is the first row of the `tightness` table above: it has the highest mean Tanimoto similarity across its five best non-self neighbors at radius 2. Accept median instead of mean if the student states the rule before comparing anchors. Do not accept “the highest single hit” because one close analog does not characterize the whole neighborhood.

3. **Radius sensitivity:** The `neighbor_movement_answer` table is the exact key for the current deterministic sample. Radius 3 represents larger atom environments, so changes are expected when two molecules share local motifs but differ in how those motifs are connected into larger substituent or scaffold contexts. Rank-only changes should be distinguished from membership changes.

4. **Scientific boundary:** Structural similarity is a hypothesis generator. It does not measure target engagement, functional activity, selectivity, exposure, toxicity, efficacy, or clinical suitability. Morgan fingerprints also compress structures into hashed features and the chosen representation may omit distinctions relevant to a particular mechanism. A repurposing claim therefore needs orthogonal biological evidence—at minimum an appropriate assay, followed by confirmation and safety/ADMET reasoning.

</details>
